Installing all libraries required for this NLP project. In machine learning and deep learning projects, libraries are extremely important as they are pre built tools and functions that make development much easier and faster. So instead of building everything from scratch, i used trusted frameworks that are already optimized for tasks like transformer training, dataset handling and evaluation.


In [ ]:
!pip install transformers datasets evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.4 MB/s eta 0:00:00


After installing the packages the next step is importing them into the notebook. Importing means bringing the functionality of these libraries into our project so we can use them in the code.

In [ ]:
import pandas as pd
import numpy as np
import torch

from datasets import load_dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, f1_score

Before moving toward dataset loading the good practice is to verify that everything is working correctly.

In [ ]:
print("Environment setup is successful!")
print("Ready dataset loading and preprocessing.")


Environment setup is successful!
Ready dataset loading and preprocessing.


Now loadng the dataset that model will learn from. Before training AI model, the important thing is to understand the data first, that what that dataset contains

using the "AG News Dataset"

This dataset contains thousands of world news headlines categorized into different topics.

using a function called load_dataset.

This function automatically:

downloads the AG News dataset from the internet
splits it into training and testing sets
organizes it into a structured format

So instead of manually downloading files, everything happens in one line.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Check Dataset Structure

In [ ]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})


Viewing Sample Data


In [ ]:
print(dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


now mapping as it helps in understandng the predictions easily, display readable output in deployment and improve UI experience later in Streamlit app

Instead of showing: “Label = 2”

Will show: “Category = Business”

In [ ]:
label_names = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

loading a pre trained tokenizer:

bert-base-uncased means:
lowercase text only
already trained on huge English corpus

This tokenizer knows:

how to split words
how to handle unknown words
how to convert words into IDs

So we don't need to build anything manually.

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

checking:

how text is broken into tokens
how token IDs are created
what format BERT receives internally

will see output like:

input_ids
attention_mask

These are essential inputs for the model.

In [ ]:
sample_text = "Apple launches new AI powered phone"

tokens = tokenizer(sample_text)
print(tokens)

{'input_ids': [101, 6207, 18989, 2047, 9932, 6113, 3042, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1]}


Now applyng the function that will do 3 important things:

1 padding="max_length"

Makes all inputs same size
Important because models need fixed length inputs

2 truncation=True

Cuts long sentences if they exceed limit
Prevents memory issues

3 max_length=128

We limit each news headline to 128 tokens
Enough for short news text

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

Applyng tokenization to dataset that takes all training + test data then applies tokenizer to each row

In [ ]:
tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset["train"][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2, 'input_ids': [101, 2813, 2358, 1012, 6468, 15020, 2067, 2046, 1996, 2304, 1006, 26665, 1007, 26665, 1011, 2460, 1011, 19041, 1010, 2813, 2395, 1005, 1055, 1040, 11101, 2989, 1032, 2316, 1997, 11087, 1011, 22330, 8713, 2015, 1010, 2024, 3773, 2665, 2153, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

Now loading BERT Model for News Classification

Now will load a pre-trained transformer model and prepare it for our specific task: news topic classification.
Using the "BERT" that is already trained on general language so im fine tuning it for our dataset (AG News)

This is called transfer learning.


BertForSequenceClassification

im not using normal BERT instead im using a version specifically designed for:
classification tasks (like sentiment or news categories)

"bert-base-uncased"
This is the base model:

already trained on huge text data
understands grammar, context, meaning

num_labels=4
This is very important as telling the model:
“must classify into 4 categories”

Those categories are:
World
Sports
Business
Sci/Tech

In [ ]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Moving model to GPU t4

In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded on:", device)

Model loaded on: cuda


In [ ]:
print(model)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Now defining how the training should happen , using Hugging Face built in training system that makes training BERT very clean and professional.
So instead of writing long manual training loops, im using : "Transformers" that provides a powerful tool called:  "Trainer API"
The Trainer Api handles training loop, backpropagation, evaluation, logging and saving checkpoints

output_dir="./results"
"Where model checkpoints will be saved"

evaluation_strategy="epoch"
"After each full training cycle (epoch) the model will be tested"

learning_rate=2e-5
"Controls how fast model learns:

too high : unstable
too low : slow learning
This value is standard for BERT.

batch_size=8
Number of samples processed at once:
higher = faster but needs more GPU memory
8 is safe for Colab GPU

num_train_epochs=2
"Model will see dataset 2 times:
enough for fine tuning
avoids overfitting
weight_decay=0.01

logging_steps=100

Print training progress every 100 steps.

save_strategy="epoch"

Model saved after each epoch automatically.

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    save_strategy="epoch"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


The cell funtion will check following:

1: Predictions vs Actual Labels
model outputs probabilities
we convert them into final class

2: Accuracy
How many predictions were correct

3: F1-score
More balanced metric and important for multi class problems

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")

    return {
        "accuracy": acc,
        "f1": f1
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    compute_metrics=compute_metrics
)

Starting the training/finetining

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.210525,0.203527,0.945263,0.945273
2,0.113611,0.213891,0.949079,0.949108


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=30000, training_loss=0.2049505305290222, metrics={'train_runtime': 6343.5047, 'train_samples_per_second': 37.834, 'train_steps_per_second': 4.729, 'total_flos': 1.578694680576e+16, 'train_loss': 0.2049505305290222, 'epoch': 2.0})

“The model achieved ~94.9% accuracy and 0.94 F1-score, indicating strong generalization on unseen news data.”

In [ ]:
save_path = "/content/drive/MyDrive/fine_tuned_bert_model_on_news"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model saved successfully at:", save_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully at: /content/drive/MyDrive/fine_tuned_bert_model_on_news


Tsting

In [ ]:
from transformers import BertForSequenceClassification, BertTokenizer

model_path = "/content/drive/MyDrive/fine_tuned_bert_model_on_news"

tokenizer = BertTokenizer.from_pretrained(model_path)
model = BertForSequenceClassification.from_pretrained(model_path)

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
label_names = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech"
}

In [ ]:
def predict_news(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    prediction = torch.argmax(logits, dim=1).item()

    return label_names[prediction]

In [ ]:
print(predict_news("Apple launches new AI powered iPhone"))

Sci/Tech


In [ ]:
print(predict_news("Manchester United wins Champions League final"))

World


In [ ]:
print(predict_news("Scientists discover new quantum computing breakthrough"))

Sci/Tech
